# Modul 2 · Logika Machine Learning secara Sederhana
**Pelatihan Machine Learning & Data Analysis**

Ide dasar Machine Learning (ML) sangat sederhana:

| | Belajar dari | Menghasilkan |
|---|---|---|
| **Manusia** | PENGALAMAN | keputusan |
| **Mesin (ML)** | DATA HISTORIS | PREDIKSI |

ML **tidak** diprogram dengan aturan "jika-maka" satu per satu. ML diberi banyak **contoh** (data historis), lalu menemukan **polanya sendiri**.

Tiga istilah kunci yang wajib dipahami:
- **FITUR (X)** — informasi yang kita ketahui (mis. bulan ke-, ada promo?)
- **TARGET (y)** — hal yang ingin diprediksi (mis. unit terjual)
- **MODEL** — "rumus pola" hasil belajar dari pasangan X dan y

> 🏅 **Aturan emas:** model diuji dengan data yang *tidak pernah ia lihat* saat belajar (data uji) — persis seperti ujian sekolah yang soalnya berbeda dari latihan.

> ⚠️ Prasyarat: notebook `00_generate_dataset.ipynb` sudah dijalankan.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

DIR = Path.cwd().parent

## LANGKAH 1 : SIAPKAN DATA HISTORIS (PENGALAMAN UNTUK BELAJAR)

In [ ]:
df = pd.read_csv(DIR / "dataset" / "data_penjualan_motor.csv")
df["urutan_bulan"] = range(1, len(df) + 1)  # bulan ke-1, ke-2, dst.

print(df[["bulan", "urutan_bulan", "ada_promo", "unit_terjual"]].head(8))
print(f"... total {len(df)} bulan data historis")

# FITUR (X) = yang kita ketahui | TARGET (y) = yang ingin diprediksi
X = df[["urutan_bulan", "no_bulan", "ada_promo"]]
y = df["unit_terjual"]

## LANGKAH 2 : BAGI DATA -> DATA LATIH vs DATA UJI

In [ ]:
# Untuk data deret waktu, pembagiannya berdasarkan URUTAN WAKTU:
# belajar dari masa lalu, diuji pada masa yang lebih baru.
batas = len(df) - 10                 # 10 bulan terakhir disisihkan untuk ujian
X_latih, X_uji = X.iloc[:batas], X.iloc[batas:]
y_latih, y_uji = y.iloc[:batas], y.iloc[batas:]

print(f"Data LATIH (untuk belajar) : {len(X_latih)} bulan pertama")
print(f"Data UJI   (untuk ujian)   : {len(X_uji)} bulan terakhir "
      f"-> model TIDAK pernah melihatnya")

## LANGKAH 3 : MODEL BELAJAR (TRAINING) - cukup 1 baris kode!

In [ ]:
model = LinearRegression()
model.fit(X_latih, y_latih)   # <- di sinilah "mesin belajar" terjadi

print("Pola yang ditemukan model dari data historis:")
for fitur, koef in zip(X.columns, model.coef_):
    print(f"  - Setiap kenaikan 1 '{fitur}' -> penjualan berubah "
          f"{koef:+,.0f} unit")
print("Contoh membaca: koefisien 'ada_promo' = perkiraan TAMBAHAN unit "
      "saat ada promo.")

## LANGKAH 4 : UJIAN! PREDIKSI DATA YANG BELUM PERNAH DILIHAT

In [ ]:
prediksi = model.predict(X_uji)

hasil = pd.DataFrame({
    "bulan": df["bulan"].iloc[batas:].values,
    "aktual": y_uji.values,
    "prediksi": prediksi.round().astype(int),
})
hasil["selisih"] = hasil["prediksi"] - hasil["aktual"]

print(hasil.to_string(index=False))

mae = mean_absolute_error(y_uji, prediksi)
rata2 = y_uji.mean()
print(f"\nRata-rata meleset (MAE)   : {mae:,.0f} unit")
print(f"Rata-rata penjualan aktual: {rata2:,.0f} unit")
print(f"Tingkat kesalahan         : {mae / rata2:.1%} "
      f"-> akurasi praktis sekitar {1 - mae / rata2:.0%}")

## LANGKAH 5 : VISUALISASI - AKTUAL vs PREDIKSI

In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(df["urutan_bulan"], y, label="Aktual (riwayat)", color="#444444")
plt.plot(X_uji["urutan_bulan"], prediksi, "o--", color="#E60012",
         label="Prediksi model (data uji)")
plt.axvline(batas + 0.5, color="gray", linestyle=":",
            label="Batas data latih | uji")
plt.title("Logika ML: Belajar dari Riwayat, Diuji pada Data Baru")
plt.xlabel("Bulan ke-")
plt.ylabel("Unit terjual")
plt.legend()
plt.tight_layout()
file_grafik = DIR / "output" / "modul2_aktual_vs_prediksi.png"
plt.savefig(file_grafik, dpi=120)
plt.show()
print(f"\nGrafik disimpan ke: {file_grafik}")

print("""
--------------------------------------------------------------------------------
KESIMPULAN MODUL 2
--------------------------------------------------------------------------------
1. ML = menemukan pola dari data historis, bukan diprogram aturan satu-satu.
2. Resep bakunya selalu sama : siapkan X dan y -> bagi latih/uji ->
   model.fit() untuk belajar -> model.predict() untuk memprediksi -> evaluasi.
3. Model HARUS diuji pada data yang belum pernah dilihat. Nilai bagus di data
   latih saja bisa menipu (istilahnya: overfitting / "menghafal soal").
4. Semua model ML lain (yang lebih canggih) tetap mengikuti logika ini.
--------------------------------------------------------------------------------
""")

---
### 💡 Latihan mandiri
1. Hapus fitur `ada_promo` dari daftar fitur, latih ulang, dan bandingkan MAE-nya — seberapa besar kontribusi informasi promo?
2. Ubah jumlah bulan data uji dari 10 menjadi 6 atau 14. Apakah kesimpulan tentang akurasi model berubah?

In [ ]:
# ---- EKSPERIMEN 1 : Tanpa fitur 'ada_promo' ----
# Bandingkan MAE dengan model sebelumnya untuk melihat kontribusi fitur promo

X_tanpa_promo = df[["urutan_bulan", "no_bulan"]]

batas = len(df) - 10
Xl_tp, Xt_tp = X_tanpa_promo.iloc[:batas], X_tanpa_promo.iloc[batas:]
yl_tp, yt_tp = y.iloc[:batas], y.iloc[batas:]

model_tp = LinearRegression()
model_tp.fit(Xl_tp, yl_tp)
pred_tp = model_tp.predict(Xt_tp)

mae_tp = mean_absolute_error(yt_tp, pred_tp)
mae_asli = mean_absolute_error(y_uji, prediksi)

print("=" * 55)
print("EKSPERIMEN 1 : Tanpa fitur 'ada_promo'")
print("=" * 55)
print(f"MAE dengan promo       : {mae_asli:>8,.0f} unit")
print(f"MAE tanpa promo        : {mae_tp:>8,.0f} unit")
print(f"Selisih (tanpa - dengan): {mae_tp - mae_asli:>+8,.0f} unit")
print()
if mae_tp > mae_asli:
    print("Kesimpulan: fitur 'ada_promo' membantu model")
    print(f"            menurunkan MAE sebesar {mae_tp - mae_asli:,.0f} unit.")
else:
    print("Kesimpulan: fitur 'ada_promo' tidak membantu banyak.")

In [ ]:
# ---- EKSPERIMEN 2 : Variasi jumlah data uji ----
# Coba 6 dan 14 bulan sebagai data uji, lihat perubahan MAE & akurasi

print("=" * 55)
print("EKSPERIMEN 2 : Variasi jumlah data uji")
print("=" * 55)

for n_uji in [6, 14]:
    batas_n = len(df) - n_uji
    Xl, Xt = X.iloc[:batas_n], X.iloc[batas_n:]
    yl, yt = y.iloc[:batas_n], y.iloc[batas_n:]

    m = LinearRegression()
    m.fit(Xl, yl)
    p = m.predict(Xt)

    mae_n = mean_absolute_error(yt, p)
    rata_yt = yt.mean()
    print(f"Data uji {n_uji:>2} bulan -> MAE: {mae_n:>7,.0f} unit  |"
          f" Akurasi: {1 - mae_n / rata_yt:.1%}")

print()
print("(Pembanding: data uji 10 bulan -> MAE:"
      f" {mae:>7,.0f} unit  | Akurasi: {1 - mae / y_uji.mean():.1%})")